In [1]:
import numpy as np
import tensorflow as tf
import keras

(X_train,y_train),(X_test,y_test) = keras.datasets.imdb.load_data(num_words=10000)
print("Reviews in training set:",len(X_train))
print("Example review (numbers):",X_train[0][:10])
print("label (1 = positive): ",y_train[0])

X_train = keras.preprocessing.sequence.pad_sequences(
    X_train,maxlen=200
)
X_test = keras.preprocessing.sequence.pad_sequences(
    X_test,maxlen=200
)
print("\nAfter padding:")
print("X_train shape:",X_train.shape)



Reviews in training set: 25000
Example review (numbers): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65]
label (1 = positive):  1

After padding:
X_train shape: (25000, 200)


In [2]:
model = keras.Sequential([
    keras.layers.Embedding(10000,32),
    keras.layers.LSTM(64),
    keras.layers.Dense(32,activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1,activation="sigmoid")
])
model.summary()
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)
history = model.fit(
    X_train,y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
test_loss,test_acc = model.evaluate(X_test,y_test,verbose=0)
print("\nTest Accuracy: ",test_acc)
model.save("Sentiment_analyzer_model.keras")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 18s 47ms/step - accuracy: 0.7639 - loss: 0.4747 - val_accuracy: 0.8680 - val_loss: 0.3244
Epoch 2/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 16s 44ms/step - accuracy: 0.8957 - loss: 0.2682 - val_accuracy: 0.8796 - val_loss: 0.3142
Epoch 3/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 16s 44ms/step - accuracy: 0.9260 - loss: 0.1992 - val_accuracy: 0.8744 - val_loss: 0.3580
Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 15s 44ms/step - accuracy: 0.9434 - loss: 0.1559 - val_accuracy: 0.8636 - val_loss: 0.3579
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 16s 44ms/step - accuracy: 0.9561 - loss: 0.1219 - val_accuracy: 0.8560 - val_loss: 0.4224

Test Accuracy:  0.8694800138473511


In [4]:
word_index = keras.datasets.imdb.get_word_index()

word_index = {w: (i+3) for w, i in word_index.items()}
import pickle
with open("word_index.pkl","wb") as f:
    pickle.dump(word_index,f)
    
word_index["<PAD>"]     = 0
word_index["<START>"]   = 1
word_index["<UNK>"]     = 2
word_index["<UNUSED>"]  = 3

def predict_sentiment(review):
  words = review.lower().split()
  
  encoded = [word_index.get(w,2) for w in words]
  encoded = [i if i < 10000 else 2 for i in encoded]

  padded = keras.preprocessing.sequence.pad_sequences(
      [encoded], maxlen=200
  )
  score = model.predict(padded,verbose=0)[0][0]
  sentiment = "POSITIVE 😊" if score > 0.5 else "NEGATIVE 😞"
  print(f"Review: {review}")
  print(f"Sentiment {sentiment} Score ({score :.2f})\n")

predict_sentiment("this movie was absolutly amazing")
predict_sentiment("this movie was terrible and boring")
predict_sentiment("i did not enjoy this film at all")
predict_sentiment("not bad actually pretty good")
predict_sentiment("very bad")
predict_sentiment("not bad")

Review: this movie was absolutly amazing
Sentiment NEGATIVE 😞 Score (0.15)

Review: this movie was terrible and boring
Sentiment NEGATIVE 😞 Score (0.01)

Review: i did not enjoy this film at all
Sentiment NEGATIVE 😞 Score (0.31)

Review: not bad actually pretty good
Sentiment NEGATIVE 😞 Score (0.04)

Review: very bad
Sentiment NEGATIVE 😞 Score (0.00)

Review: not bad
Sentiment NEGATIVE 😞 Score (0.00)



In [9]:
from transformers import pipeline

# default model
classifier = pipeline("sentiment-analysis")

reviews = [
    "this movie was absolutely amazing",
    "this movie was terrible and boring",
    "i did not enjoy this film at all",
    "not bad actually pretty good"
]
model.save("Sentiment_analyzer_model.keras")

for review in reviews:
    result = classifier(review)[0]
    print(f"Review: {review}")
    print(f"Result: {result['label']} ({result['score']:.2%})\n")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


Review: this movie was absolutely amazing
Result: POSITIVE (99.99%)

Review: this movie was terrible and boring
Result: NEGATIVE (99.98%)

Review: i did not enjoy this film at all
Result: NEGATIVE (99.82%)

Review: not bad actually pretty good
Result: POSITIVE (99.95%)

